# Definizione del Progetto "Plate Hunter"

## Funzionalità principali (MVP)
1. **Setup Partenza:** schermata per inserire i nomi dei due (o più) giocatori.

2. **Dashboard di Gioco:**
    - Grandi pulsanti con le bandiere dei paesi più comuni (es. Germania, Francia, Svizzera, Austria, ecc.).
    - Un pulsante "Altro" per inserire manualmente un paese raro.
    - Selettore del giocatore attivo (chi ha gridato per primo?).

3. **Tabellone Punti:** visualizzazione in tempo reale dei punteggi.

4. **Cronologia:** una lista a scorrimento (es: "14:20 - Germania (Giocatore A)") per evitare litigi su chi ha preso cosa.

5. **Persistenza:** all'apertura, l'app controlla se c'è una partita in corso nel `localStorage` e la ricarica.

## Come potrebbe funzionare tecnicamente

### La struttura dati: 
Invece di singole variabili, salveremo nel `localStorage` un unico oggetto JSON:

In [ ]:
const partita = {
    giocatori: [{nome: "Tu", punti: 5}, {nome: "Lui", punti: 3}],
    cronologia: [
        {paese: "Germania", ora: "15:30", autore: "Tu"},
        {paese: "Francia", ora:"15:35", autore: "Lui"}
    ],
    dataInizio: "2026-04-16"
};

// per salvare: localStorage.setItem('partitaCorrente', JSON.stringify(partita));

## The "Tier-Based" Scoring System
Invece di calcolare la distanza dinamica (che sarebbe complesso senza API esterne), useremo una **classifica a fasce (Tiers)**. È più facile da gestire e molto più bilanciato per il gameplay.
- **Tier 1: Neighbors (1 Point)** - Paesi confinanti con l'Italia o con turismo importante (es: Francia, Germania, Svizzera, Austria).
- **Tier 2: The Travelers (3 Points)** - un po' più lontani, ma abbastanza comuni (es: Spagna, Paesi Bassi, Polonia, Belgio).
- **Tier 3: The Long Haul (5 Points)** - lontani o meno comuni (es: Portogallo, Svezia, Ucraina, Grecia).
- **Tier 4: The Unicorns (10 Points)** - micro-stati o molto rari (es: San Marino, Città del Vaticano, Monaco o targhe non europee).

## Struttura dati (Il "Plate Dictionary")

In [ ]:
const plateDictionary = [
  // TIER 1: Neighbors & Very Common (1 Point)
  { id: "IT", name: "Italy", flag: "🇮🇹", points: 1, tier: 1 },
  { id: "DE", name: "Germany", flag: "🇩🇪", points: 1, tier: 1 },
  { id: "FR", name: "France", flag: "🇫🇷", points: 1, tier: 1 },
  { id: "CH", name: "Switzerland", flag: "🇨🇭", points: 1, tier: 1 },
  { id: "AT", name: "Austria", flag: "🇦🇹", points: 1, tier: 1 },

  // TIER 2: Common Travelers (3 Points)
  { id: "ES", name: "Spain", flag: "🇪🇸", points: 3, tier: 2 },
  { id: "NL", name: "Netherlands", flag: "🇳🇱", points: 3, tier: 2 },
  { id: "BE", name: "Belgium", flag: "🇧🇪", points: 3, tier: 2 },
  { id: "PL", name: "Poland", flag: "🇵🇱", points: 3, tier: 2 },
  { id: "RO", name: "Romania", flag: "🇷🇴", points: 3, tier: 2 },
  { id: "HR", name: "Croatia", flag: "🇭🇷", points: 3 , tier: 2 },
  { id: "SI", name: "Slovenia", flag: "🇸🇮", points: 3, tier: 2 },
  { id: "HU", name: "Hungary", flag: "HU", points: 3, tier: 2 },

  // TIER 3: The Long Haul (5 Points)
  { id: "GB", name: "United Kingdom", flag: "🇬🇧", points: 5, tier: 3 },
  { id: "PT", name: "Portugal", flag: "🇵🇹", points: 5, tier: 3 },
  { id: "SE", name: "Sweden", flag: "🇸🇪", points: 5, tier: 3 },
  { id: "GR", name: "Greece", flag: "🇬🇷", points: 5, tier: 3 },
  { id: "UA", name: "Ukraine", flag: "🇺🇦", points: 5, tier: 3 },
  { id: "AL", name: "Albania", flag: "🇦🇱", points: 5, tier: 3 },
  { id: "BA", name: "Bosnia & Herzegovina", flag: "🇧🇦", points: 5, tier: 3 },
  { id: "BG", name: "Bulgaria", flag: "🇧🇬", points: 5, tier: 3 },
  { id: "CZ", name: "Czechia", flag: "🇨🇿", points: 5, tier: 3 },
  { id: "DK", name: "Denmark", flag: "🇩🇰", points: 5, tier: 3 },
  { id: "IE", name: "Ireland", flag: "🇮🇪", points: 5, tier: 3 },
  { id: "LV", name: "Latvia", flag: "🇱🇻", points: 5, tier: 3 },
  { id: "MK", name: "North Macedonia", flag: "🇲🇰", points: 5, tier: 3 },
  { id: "MD", name: "Moldova", flag: "🇲🇩", points: 5, tier: 3 },
  { id: "RU", name: "Russia", flag: "🇷🇺", points: 5, tier: 3 },
  { id: "RS", name: "Serbia", flag: "🇷🇸", points: 5, tier: 3 },
  { id: "SK", name: "Slovakia", flag: "🇸🇰", points: 5, tier: 3 },
  { id: "TR", name: "Turkiye", flag: "🇹🇷", points: 5, tier: 3 },

  // TIER 4: The Unicorns (10 Points)
  { id: "SM", name: "San Marino", flag: "🇸🇲", points: 10, tier: 4 },
  { id: "VA", name: "Vatican City", flag: "🇻🇦", points: 10, tier: 4 },
  { id: "MC", name: "Monaco", flag: "🇲🇨", points: 10, tier: 4 },
  { id: "IS", name: "Iceland", flag: "🇮🇸", points: 10, tier: 4 },
  { id: "AD", name: "Andorra", flag: "🇦🇩", points: 10, tier: 4 },
  { id: "AM", name: "Armenia", flag: "🇦🇲", points: 10, tier: 4 },
  { id: "AZ", name: "Azerbaigian", flag: "🇦🇿", points: 10, tier: 4 },
  { id: "BY", name: "Belarus", flag: "🇧🇾", points: 10, tier: 4 },
  { id: "CY", name: "Cyprus", flag: "🇨🇾", points: 10, tier: 4 },
  { id: "EE", name: "Estonia", flag: "🇪🇪", points: 10 , tier: 4 },
  { id: "FI", name: "Finland", flag: "🇫🇮", points: 10, tier: 4 },
  { id: "GE", name: "Georgia", flag: "🇬🇪", points: 10, tier: 4 },
  { id: "KZ", name: "Kazakhstan", flag: "🇰🇿", points: 10, tier: 4 },
  { id: "LI", name: "Liechtenstein", flag: "🇱🇮", points: 10, tier: 4 },
  { id: "LT", name: "Lithuania", flag: "🇱🇹", points: 10, tier: 4 },
  { id: "LU", name: "Luxembourg", flag: "🇱🇺", points: 10, tier: 4 },
  { id: "MT", name: "Malta", flag: "🇲🇹", points: 10, tier: 4 },
  { id: "ME", name: "Montenegro", flag: "🇲🇪", points: 10, tier: 4 },
  { id: "NO", name: "Norway", flag: "🇳🇴", points: 10, tier: 4 },
];

## Come manipolare questi dati per la UI
Dato che i paesi sono molti, l'utente non può scorrerli ogni volta.
- **Quick Access:** mostra i 5-6 paesi del **Tier 1** come bottoni grandi in alto.
- **Search/Grid:** metti gli altri in una griglia o in un menu a tendina con ricerca per nome.
- **Color-Coding:** puoi usare dei colori diversi per i bottoni in base al punteggio (es: grigio per 1pt, Oro per 10pt).

### TODO:
- *trasformare la top9 in una top11 (così il tasto more ne ha due affianco) + togliere Italia + spagna in tier 1*
- *aggiungere ID nei pulsanti così da non avere solo la bandiera*
- *correggere verso swipe*
- *cambiare i colori dei punteggi nella lista ricerca (in base ai tier)*
- *sarebbe carino che restassero fissi i tier 1 e gli altri bottoni variassero dinamicamente in base alla cronologia (tipo inizialmente sono i primi tier 2 e poi in qualche modo variassero per mostrare gli ultimi chiamati così da averli sotto mano, come una specie di coda)*
- *sotto i punteggi aggiungere in piccolo l'ultima chiamata (es: Marta: bandiera germania (orario))*
- *capire history (partite precedenti) e lista "chiamate"*
- *non vanno i tasti UNDO, RESET TRIP, HISTORY*

- evidenziare vincitore nella history + togliere past trips summary ripetuto
- font più grande
- evidenziare players

- aggiornare readme

- info how to play (in alto e allo start)
- statistiche stati globale?
